<a href="https://colab.research.google.com/github/adljna/ProjectA-Group3-KematianAliKhamenei/blob/main/Article%20Link%20Scrapping/Kompas/1_Kompas_ScrappingLink.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Scrapping Berita Kematian Ali Khamenei - Kompas**
Notebook ini berisi proses scraping berita terkait 'Kematian Ali Khamenei' dari situs Kompas.com. Prosesnya meliputi inisialisasi variabel, ekstraksi data per halaman, looping untuk mengumpulkan artikel hingga target tercapai (100 artikel), dan penyimpanan data hasil scraping ke dalam format CSV menggunakan Pandas.

# **(1) Import Library**

Bagian ini berisi semua import library Python yang dibutuhkan untuk operasi scraping dan manipulasi data.

In [1]:
from bs4 import BeautifulSoup
import requests
import time
import pandas as pd

# **(2) Inisialisasi Variabel Global**

Bagian ini mendefinisikan variabel global seperti `headers` untuk permintaan HTTP (mencegah pemblokiran) dan `target` jumlah artikel yang akan discrape.

In [6]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

# target jumlah artikel
target = 100

# **(3) Fungsi Ekstraksi Data Per Halaman**

Fungsi `extract_data_from_page` bertanggung jawab untuk mengambil data (judul, link, penerbit, tanggal) dari satu halaman pencarian di Detik.com. Fungsi ini menggunakan `BeautifulSoup` untuk mem-parsing HTML dan mengekstrak informasi yang relevan.

In [7]:
def extract_data_from_page(page_num):
    url = f"https://search.kompas.com/search?q=Ali+Khamenei+death&site_id=all&last_date=all&page={page_num}"

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Gagal mengambil halaman {page_num}")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    articles = soup.find_all("div", class_="articleItem")

    data = []

    for article in articles:
        # Link (parent utama)
        link_tag = article.find("a", class_="article-link")
        link = link_tag["href"] if link_tag else None

        # Judul
        title_tag = article.find("h2", class_="articleTitle")
        title = title_tag.get_text(strip=True) if title_tag else None

        # Tanggal
        date_tag = article.find("div", class_="articlePost-date")
        date = date_tag.get_text(strip=True) if date_tag else None

        # Kategori (opsional, kalau mau)
        category_tag = article.find("div", class_="articlePost-subtitle")
        category = category_tag.get_text(strip=True) if category_tag else None

        # Penerbit
        publisher = "Kompas"

        data.append({
            "Judul": title,
            "Penerbit": publisher,
            "Kategori": category,
            "Link": link,
            "Tanggal Published": date
        })

    return data

# **(4) Proses Scraping Utama (Looping)**

Bagian ini mengelola proses scraping utama, melakukan loop melalui halaman-halaman pencarian Detik.com hingga mencapai target jumlah artikel yang diinginkan (100 artikel). Terdapat jeda 2 detik antar permintaan (`time.sleep(2)`) untuk menghindari pemblokiran IP dan memberikan waktu bagi server.

In [8]:
all_data = []
page = 1

while len(all_data) < target:

    print(f"Scraping halaman {page}")

    page_data = extract_data_from_page(page)

    if not page_data:
        break

    all_data.extend(page_data)

    page += 1
    time.sleep(2)

# ambil hanya 100 artikel sesuai target
all_data = all_data[:target]

Scraping halaman 1
Scraping halaman 2
Scraping halaman 3
Scraping halaman 4
Scraping halaman 5


# **(5) Pembentukan DataFrame dan Penyimpanan Data**

Setelah data berhasil dikumpulkan, bagian ini mengubahnya menjadi DataFrame Pandas untuk memudahkan analisis. Kemudian, DataFrame tersebut disimpan ke dalam file CSV dengan nama `detik_khamenei_articles.csv`.

In [9]:
# Buat DataFrame dari data yang dikumpulkan
df = pd.DataFrame(all_data)

# simpan ke CSV (Format: publisher_khamenei_articles.csv)
df.to_csv("kompas_khamenei_articles.csv", index=False, encoding="utf-8")

print("Total artikel:", len(df))
print("CSV berhasil dibuat!")

Total artikel: 100
CSV berhasil dibuat!
